# 10. Business Optimization

**Objective:**  
Optimize the trained model for real-time sign language recognition by:
- Applying confidence thresholding
- Reducing false predictions
- Improving reliability for end-user communication


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.metrics import accuracy_score, f1_score


In [2]:
MODEL_PATH = Path("../models/final/random_forest_tuned.pkl")
TEST_PATH  = Path("../data/test_fe.csv")

model = joblib.load(MODEL_PATH)
test_df = pd.read_csv(TEST_PATH)

X_test = test_df.drop(columns=["class"])
y_test = test_df["class"]

print("✅ Model and test data loaded")
print("Test samples:", X_test.shape[0])


✅ Model and test data loaded
Test samples: 3283


In [3]:
y_probs = model.predict_proba(X_test)
y_pred_default = np.argmax(y_probs, axis=1)

print("✅ Prediction probabilities computed")


✅ Prediction probabilities computed


In [4]:
THRESHOLD = 0.70

y_pred_thresholded = []
rejected = 0

for prob in y_probs:
    if np.max(prob) >= THRESHOLD:
        y_pred_thresholded.append(np.argmax(prob))
    else:
        y_pred_thresholded.append(-1)  # reject prediction
        rejected += 1

y_pred_thresholded = np.array(y_pred_thresholded)

print(f"🚫 Rejected predictions (low confidence): {rejected}")
print(f"Accepted predictions: {len(y_test) - rejected}")


🚫 Rejected predictions (low confidence): 241
Accepted predictions: 3042


In [5]:
valid_idx = y_pred_thresholded != -1

acc = accuracy_score(y_test[valid_idx], y_pred_thresholded[valid_idx])
f1  = f1_score(y_test[valid_idx], y_pred_thresholded[valid_idx], average="weighted")

print("📊 Performance after thresholding:")
print(f"Accuracy: {acc:.3f}")
print(f"F1-score: {f1:.3f}")


📊 Performance after thresholding:
Accuracy: 1.000
F1-score: 1.000


In [6]:
print("📌 Business Optimization Summary:")
print("- Low-confidence predictions are safely rejected")
print("- False activations reduced")
print("- Improves trust in real-time sign language output")


📌 Business Optimization Summary:
- Low-confidence predictions are safely rejected
- False activations reduced
- Improves trust in real-time sign language output


In [7]:
OUTPUT_DIR = Path("../reports/tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame({
    "accuracy_after_threshold": [acc],
    "f1_after_threshold": [f1],
    "rejected_samples": [rejected]
}).to_csv(OUTPUT_DIR / "business_optimization_results.csv", index=False)

print("✅ Business optimization results saved")


✅ Business optimization results saved
